# 04 - Equity and Fairness Analysis

CRISP-DM stage covered: Evaluation with external context (population-adjusted access).

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loading import load_clean_hospital_data
from src.features import add_basic_capacity_features

In [2]:
df = load_clean_hospital_data()
state_facilities = (
    df.groupby('STATE')
    .agg(
        facilities=('STATE', 'size'),
        total_beds=('TOTAL_BEDS', 'sum'),
        burn_beds=('BURN_BEDS', 'sum'),
    )
    .reset_index()
)

pop_path = ROOT / 'data' / 'external' / 'state_population_2012.csv'
pop_df = pd.read_csv(pop_path)

equity_df = state_facilities.merge(pop_df, on='STATE', how='left')
equity_df = add_basic_capacity_features(equity_df, population_col='population')

equity_df[['STATE', 'population', 'total_beds_per_100k', 'burn_beds_per_100k']].sort_values('total_beds_per_100k', ascending=False).head(20)

,STATE,population,total_beds_per_100k,burn_beds_per_100k
7,DC,633427.0,303.113066,6.472727
28,ND,701345.0,207.173360,0.000000
6,CT,3591765.0,185.201426,0.250573
41,SD,834047.0,180.685261,0.000000
22,MI,9882519.0,170.725703,0.698203
24,MO,6024522.0,159.431736,1.543691
14,IL,12868192.0,141.884734,0.598375
8,DE,917053.0,139.250403,0.000000
49,WV,1856680.0,138.149816,0.215438
26,MT,1005494.0,134.958538,0.000000


In [3]:
low_access = equity_df[['STATE', 'total_beds_per_100k', 'burn_beds_per_100k']].sort_values('total_beds_per_100k').head(10)
low_access

,STATE,total_beds_per_100k,burn_beds_per_100k
32,NM,26.685353,0.479952
36,OK,33.125599,0.864830
47,WA,43.188726,0.580104
20,MD,45.421580,0.339855
17,KY,45.596418,0.365319
4,CA,59.347559,0.368422
42,TN,64.121691,0.790096
31,NJ,68.078156,0.135322
39,RI,68.456371,0.000000
37,OR,70.977981,0.410277
